# 06 — Caliper pipeline

This notebook walks through the caliper analysis pipeline as it now
lives in ``karst_analysis.caliper``.

## Convention

Depths are **BGL-positive** throughout: 0 = ground level, deeper =
larger positive numbers. This matches the original LAS files and the
SEC sub-package. The word "Elevation" stays reserved for the day
differential GPS data lands (absolute height above sea level).

## Pipeline steps

1. Load the master concatenated caliper CSV.
2. Estimate instrumental noise (AW5O smooth vs AW5D rotary auger).
3. Run cumulative-min baseline + breakout detection on each priority well.
4. Inspect the per-sample classification.
5. Render the multi-well panel.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import json
import pandas as pd
import matplotlib.pyplot as plt

from karst_analysis.caliper import (
    load_master_caliper,
    estimate_noise_aw5o_vs_aw5d,
    process_many_wells,
    perpoint_dataframe,
    zones_dataframe,
    plot_priority_wells_panel,
)
from karst_analysis.caliper.config import PRIORITY_WELLS

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print(f"CWD: {Path.cwd()}")
print(f"Priority wells: {PRIORITY_WELLS}")

## 1. Load master caliper

Note: the CSV has a column called ``Depth [m]`` on disk; the loader
renames it to ``depth_m`` (snake_case) and verifies BGL-positive
values. If you see a "negative depths" error here, run
``scripts/fix_caliper_master_signs.py`` once to fix the file.

In [ ]:
df = load_master_caliper()
print(f"Shape: {df.shape}")
print(f"depth_m range: [{df['depth_m'].min():.3f}, {df['depth_m'].max():.3f}]")
df.head()

## 2. Noise estimation

The detection threshold uses ``sigma_inst`` measured on AW5O (smooth
sonic-cored hole). The script estimates it from a fixed depth interval
``[1, 5]`` m BGL and writes a JSON report.

In [ ]:
noise = estimate_noise_aw5o_vs_aw5d(df)
sigma = noise["AW5O"]["sigma_MAD_cm"]
print(f"sigma_inst (AW5O sigma_MAD) = {sigma:.6f} cm")
print(f"sigma_drilling (AW5D - AW5O, from MAD) = {noise['comparison']['sigma_drilling_from_MAD_cm']:.4f} cm")

## 3. Run pipeline on the 5 priority wells

In [ ]:
results = process_many_wells(df, sigma)

for w, r in results.items():
    sev_counts = pd.Series(r["perpoint"]["severity"]).value_counts().to_dict()
    print(f"  {w}: {len(r['zones']):>2} zones, severity={sev_counts}")

## 4. Per-sample DataFrame for downstream consumption

In [ ]:
perpoint = perpoint_dataframe(results)
print(f"Shape: {perpoint.shape}")
print(f"depth_m range: [{perpoint['depth_m'].min():.3f}, {perpoint['depth_m'].max():.3f}]")
perpoint.head(10)

## 5. Multi-well panel figure

In [ ]:
fig_path = plot_priority_wells_panel(
    results, sigma,
    output_path="results/figures/caliper/priority_wells_cumulative_min_v2_panel.png",
)
print(f"Saved: {fig_path}")

from IPython.display import Image
Image(filename=str(fig_path))